# Jersey -> Celtics Jersey CycleGAN by John Lunardini

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Phase 1: Data Preparation

In [2]:
from ultralytics import YOLO
from PIL import Image
from pathlib import Path
import random, shutil

ROOT = Path("/content/drive/MyDrive/cycleGAN_celtics_project/content")
model = YOLO("yolo11n-pose.pt")
TORSO = [5, 6, 11, 12]            # COCO keypoints: left and right shoulders and hips

def crop_torsos(src, dst, size=256):
    dst.mkdir(parents=True, exist_ok=True)
    kept = 0
    for p in sorted(src.iterdir()):
        if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        img = Image.open(p).convert("RGB")
        r = model(img, verbose=False)[0]
        if len(r.boxes) == 0:
            continue
        i = (r.boxes.xywh[:, 2] * r.boxes.xywh[:, 3]).argmax().item()  #biggest person
        kp = r.keypoints.xy[i].cpu().numpy()
        conf = r.keypoints.conf[i].cpu().numpy()
        if (conf[TORSO] < 0.5).any():   # skip if shoulders and hips not clearly visible
            continue
        xs, ys = kp[TORSO, 0], kp[TORSO, 1]
        cx, cy = xs.mean(), ys.mean()
        side = max((xs.max() - xs.min()) * 1.6, (ys.max() - ys.min()) * 1.4)
        box = (cx - side/2, cy - side/2, cx + side/2, cy + side/2)
        img.crop(box).resize((size, size), Image.LANCZOS).save(dst / f"{p.stem}.jpg", quality=95)
        kept += 1
    print(f"{src.name}: kept {kept}")

#crop_torsos(ROOT / "raw_A", ROOT / "tmp_A")
#crop_torsos(ROOT / "raw_B", ROOT / "tmp_B")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


Don't rerun the rest of phase 1

In [3]:
"""
!pip install -q transformers
from transformers import CLIPModel, CLIPProcessor
import torch, shutil

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").cuda().eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
prompts = [
    "a basketball player wearing a green Boston Celtics jersey",   # the one we want
    "a basketball player wearing a white Boston Celtics jersey",
    "a basketball player wearing another team's jersey",
    "a photo of a crowd, coach, or fan, not a player",
]

def clip_filter(src, keep, maybe, hi=0.7, lo=0.4):
    keep.mkdir(parents=True, exist_ok=True); maybe.mkdir(parents=True, exist_ok=True)
    paths = sorted(src.glob("*.jpg"))
    for i in range(0, len(paths), 64):
        batch = paths[i:i+64]
        imgs = [Image.open(p).convert("RGB") for p in batch]
        inp = proc(text=prompts, images=imgs, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            p_green = clip(**inp).logits_per_image.softmax(-1)[:, 0].cpu()
        for path, s in zip(batch, p_green.tolist()):
            if s >= hi:   shutil.copy(path, keep / path.name)
            elif s >= lo: shutil.copy(path, maybe / path.name)
    print(f"keep: {len(list(keep.glob('*.jpg')))}, maybe: {len(list(maybe.glob('*.jpg')))}")
"""
#clip_filter(ROOT / "tmp_B", ROOT / "clip_keep_B", ROOT / "clip_maybe_B")

'\n!pip install -q transformers\nfrom transformers import CLIPModel, CLIPProcessor\nimport torch, shutil\n\nclip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").cuda().eval()\nproc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")\nprompts = [\n    "a basketball player wearing a green Boston Celtics jersey",   # the one we want\n    "a basketball player wearing a white Boston Celtics jersey",\n    "a basketball player wearing another team\'s jersey",\n    "a photo of a crowd, coach, or fan, not a player",\n]\n\ndef clip_filter(src, keep, maybe, hi=0.7, lo=0.4):\n    keep.mkdir(parents=True, exist_ok=True); maybe.mkdir(parents=True, exist_ok=True)\n    paths = sorted(src.glob("*.jpg"))\n    for i in range(0, len(paths), 64):\n        batch = paths[i:i+64]\n        imgs = [Image.open(p).convert("RGB") for p in batch]\n        inp = proc(text=prompts, images=imgs, return_tensors="pt", padding=True).to("cuda")\n        with torch.no_grad():\n            p_green = 

In [4]:
def split(tmp, name, test_frac=0.1):
    files = sorted(tmp.glob("*.jpg"))
    random.seed(0); random.shuffle(files)
    n_test = int(len(files) * test_frac)
    for subset, chunk in [("test", files[:n_test]), ("train", files[n_test:])]:
        out = ROOT / "data" / f"{subset}{name}"
        out.mkdir(parents=True, exist_ok=True)
        for f in chunk:
            shutil.copy(f, out / f.name)

#split(ROOT / "tmp_A", "A")
#split(ROOT / "tmp_B", "B")

Phase 2: Building CycleGAN

In [5]:
# setup

import os, random, itertools, shutil
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.utils import save_image
from PIL import Image

ROOT = Path("/content/drive/MyDrive/cycleGAN_celtics_project/content")

# copy the dataset to local disk for faster loading
shutil.copytree(ROOT / "data", "/content/data", dirs_exist_ok=True)
(ROOT / "checkpoints").mkdir(exist_ok=True)

In [6]:
# configuration of the hyperparameters
@dataclass
class Config:
    # paths
    dataroot: str = "/content/data"                        # local copy of data
    name: str = "celtics_cyclegan"
    checkpoints_dir: str = str(ROOT / "checkpoints")       # on drive so it survives disconnects

    # model
    input_nc: int = 3
    output_nc: int = 3
    ngf: int = 64              # generator base width
    ndf: int = 64              # discriminator base width
    n_res_blocks: int = 9
    init_gain: float = 0.02    # std of the starting weights

    # data
    load_size: int = 286       #resize
    crop_size: int = 256       #crop
    no_flip: bool = False      # false = random horizontal flips on
    batch_size: int = 1
    num_workers: int = 2

    # training
    lr: float = 0.0002
    beta1: float = 0.5         #Adam momentum term
    n_epochs: int = 25        # epochs at constant lr
    n_epochs_decay: int = 25  # epochs where lr decays linearly to 0
    lambda_A: float = 10.0     # cycle loss weight, A -> B -> A
    lambda_B: float = 10.0     # cycle loss weight, B -> A -> B
    lambda_identity: float = 0.5   #identity loss = 0.5 * lambda
    pool_size: int = 50

    #logging/saving
    print_freq: int = 100
    save_epoch_freq: int = 5
    continue_train: bool = True   # true = resume from the latest checkpoint
    epoch_count: int = 1

    #testing
    results_dir: str = str(ROOT / "results")
    load_epoch: str = "latest"
    num_test: int = 50

DEBUG = False
if DEBUG:
    cfg = Config(load_size=143, crop_size=128, n_res_blocks=6,
                 n_epochs=2, n_epochs_decay=2, save_epoch_freq=1)
else:
    cfg = Config()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(cfg, device)

Config(dataroot='/content/data', name='celtics_cyclegan', checkpoints_dir='/content/drive/MyDrive/cycleGAN_celtics_project/content/checkpoints', input_nc=3, output_nc=3, ngf=64, ndf=64, n_res_blocks=9, init_gain=0.02, load_size=286, crop_size=256, no_flip=False, batch_size=1, num_workers=2, lr=0.0002, beta1=0.5, n_epochs=25, n_epochs_decay=25, lambda_A=10.0, lambda_B=10.0, lambda_identity=0.5, pool_size=50, print_freq=100, save_epoch_freq=5, continue_train=True, epoch_count=1, results_dir='/content/drive/MyDrive/cycleGAN_celtics_project/content/results', load_epoch='latest', num_test=50) cpu


In [7]:
# dataset class
# dataloader

class UnpairedDataset(Dataset):
  def __init__(self, cfg, phase = "train"):
    # folders for this phase
    self.dir_A = Path(cfg.dataroot) /f"{phase}A"
    self.dir_B = Path(cfg.dataroot) /f"{phase}B"

    #sorted lists of image paths in each folder (sorted so order is repeatable)
    exts = {".jpg",".jpeg",".png"}
    self.A_paths = sorted(p for p in self.dir_A.iterdir() if p.suffix.lower() in exts)
    self.B_paths = sorted(p for p in self.dir_B.iterdir() if p.suffix.lower() in exts)

    bicubic = T.InterpolationMode.BICUBIC
    if phase == "train":
      steps = [T.Resize((cfg.load_size, cfg.load_size), interpolation = bicubic), T.RandomCrop(cfg.crop_size)]
      if not cfg.no_flip:
        steps.append(T.RandomHorizontalFlip()) #50% chance of a mirror image
    else:
      steps = [T.Resize((cfg.crop_size, cfg.crop_size), interpolation = bicubic)]
    steps += [T.ToTensor(), T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
    self.transform = T.Compose(steps)

  def __len__(self):
    return max(len(self.A_paths), len(self.B_paths))

  def __getitem__(self, index):
    A_path = self.A_paths[index % len(self.A_paths)]
    B_path = self.B_paths[random.randint(0, len(self.B_paths) - 1)]
    A_img = Image.open(A_path).convert("RGB")
    B_img = Image.open(B_path).convert("RGB")
    return {"A": self.transform(A_img), "B" : self.transform(B_img)}



In [8]:
# generator

class ResnetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size=3),
            nn.InstanceNorm2d(dim),
            nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size=3),
            nn.InstanceNorm2d(dim),
        )

    def forward(self, x):
        return x + self.block(x)   #keep x, add only the learned change

class Generator(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ngf = cfg.ngf

        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(cfg.input_nc, ngf, kernel_size=7),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(True),
        ]

        # downsample twice
        ch = ngf
        for _ in range(2):
            layers += [
                nn.Conv2d(ch, ch * 2, kernel_size=3, stride=2, padding=1),
                nn.InstanceNorm2d(ch * 2),
                nn.ReLU(True)
            ]
            ch *= 2


        layers += [ResnetBlock(ch) for _ in range(cfg.n_res_blocks)]
        #upsample twice
        for _ in range(2):
          layers += [
              nn.ConvTranspose2d(ch, ch // 2, kernel_size=3, stride=2, padding=1, output_padding = 1),
              nn.InstanceNorm2d(ch // 2),
              nn.ReLU(True)
          ]
          ch //=2
        # output layer
        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, cfg.output_nc, kernel_size=7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [9]:
# discriminator

class Discriminator(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    ndf = cfg.ndf
    layers = [
        nn.Conv2d(cfg.input_nc, ndf, kernel_size=4, stride=2, padding=1),
        nn.LeakyReLU(0.2, True)
    ]
    ch = ndf
    #downsample twice
    for _ in range(2):
      layers += [
              nn.Conv2d(ch, ch * 2, kernel_size=4, stride=2, padding=1),
              nn.InstanceNorm2d(ch*2),
              nn.LeakyReLU(0.2, True)
          ]
      ch *=2
    layers += [
              nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1),
              nn.InstanceNorm2d(512),
              nn.LeakyReLU(0.2, True)
          ]
    layers+= [
              nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)]
    self.model = nn.Sequential(*layers)
  def forward(self, x):
    return self.model(x)

In [10]:
# losses cell

criterion_GAN = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_idt = nn.L1Loss()

def gan_loss(pred, target_is_real):
  if target_is_real:
    target = torch.ones_like(pred)
  else:
    target = torch.zeros_like(pred)
  return criterion_GAN(pred, target)

In [11]:
# image buffer cell

class ImagePool:
  def __init__(self, pool_size):
    self.images = []
    self.pool_size = pool_size
  def query(self, images):
    if self.pool_size == 0:
      return images

    result = []
    for image in images:
      image = image.unsqueeze(0).detach()

      if len(self.images) < self.pool_size:
        self.images.append(image)
        result.append(image)
      else:
        if random.random() < 0.5:
          i = random.randint(0, self.pool_size-1)
          result.append(self.images[i].clone())
          self.images[i]= image
        else:
          result.append(image)
    return torch.cat(result, dim=0)

In [12]:
# build models, optimizers, schedulers

G_AB = Generator(cfg).to(device) #other -> celtics
G_BA = Generator(cfg).to(device) #celtics -> other
D_A = Discriminator(cfg).to(device) # is this a real other image
D_B = Discriminator(cfg).to(device) #is this a real celtics image

def init_weights(m):
  if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
    nn.init.normal_(m.weight, 0.0, cfg.init_gain)
    if m.bias is not None:
      nn.init.zeros_(m.bias)

  G_AB.apply(init_weights)
  G_BA.apply(init_weights)
  D_A.apply(init_weights)
  D_B.apply(init_weights)

opt_G = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()),
                         lr=cfg.lr, betas=(cfg.beta1, 0.999))
opt_D = torch.optim.Adam(itertools.chain(D_A.parameters(), D_B.parameters()),
                         lr=cfg.lr, betas=(cfg.beta1,0.999))

def lr_lambda(epoch):
    return 1.0 - max(0, epoch + cfg.epoch_count - cfg.n_epochs) / float(cfg.n_epochs_decay + 1)

sched_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_lambda=lr_lambda)
sched_D = torch.optim.lr_scheduler.LambdaLR(opt_D, lr_lambda=lr_lambda)

fake_A_pool = ImagePool(cfg.pool_size)
fake_B_pool = ImagePool(cfg.pool_size)

Phase 3: Training

In [13]:
# checkpoints

def save_checkpoint(epoch):
    ckpt = {
        "epoch": epoch,
        "G_AB": G_AB.state_dict(),
        "G_BA": G_BA.state_dict(),
        "D_A": D_A.state_dict(),
        "D_B": D_B.state_dict(),
        "opt_G": opt_G.state_dict(),
        "opt_D": opt_D.state_dict(),
        "sched_G": sched_G.state_dict(),
        "sched_D": sched_D.state_dict(),
    }
    torch.save(ckpt, Path(cfg.checkpoints_dir) / "latest.pth")
    if epoch % cfg.save_epoch_freq == 0:
        torch.save(G_AB.state_dict(), Path(cfg.checkpoints_dir) / f"G_AB_epoch{epoch}.pth")

def load_checkpoint():
  path = Path(cfg.checkpoints_dir) / "latest.pth"
  if not path.exists():
    print("Starting at epoch 1")
    return 1

  ckpt = torch.load(path, map_location = device)

  # copy the saved numbers back into each object
  G_AB.load_state_dict(ckpt["G_AB"])
  G_BA.load_state_dict(ckpt["G_BA"])
  D_A.load_state_dict(ckpt["D_A"])
  D_B.load_state_dict(ckpt["D_B"])
  opt_G.load_state_dict(ckpt["opt_G"])
  opt_D.load_state_dict(ckpt["opt_D"])
  sched_G.load_state_dict(ckpt["sched_G"])
  sched_D.load_state_dict(ckpt["sched_D"])

  print(f"Resumed from epoch {ckpt['epoch']}")
  return ckpt["epoch"] + 1    # the next epoch to train

In [14]:
# training loop

test_ds = UnpairedDataset(cfg, phase="test")
fixed_A = torch.stack([test_ds[i]["A"] for i in range(4)]).to(device)   # 4 fixed test images

def set_requires_grad(nets, flag):
    for net in nets:
        for p in net.parameters():
            p.requires_grad = flag

train_ds = UnpairedDataset(cfg, phase="train")
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)

sample_dir = ROOT / "samples"
sample_dir.mkdir(exist_ok=True)

start_epoch = load_checkpoint() if cfg.continue_train else 1
total_epochs = cfg.n_epochs + cfg.n_epochs_decay

for epoch in range(start_epoch, total_epochs + 1):
    for i, batch in enumerate(train_loader):
        real_A = batch["A"].to(device)
        real_B = batch["B"].to(device)

        # forward passes
        fake_B = G_AB(real_A)     # other -> Celtics
        rec_A  = G_BA(fake_B)     # back to other
        fake_A = G_BA(real_B)     # Celtics -> other
        rec_B  = G_AB(fake_A)     # back to Celtics

        # generator update
        set_requires_grad([D_A, D_B], False)
        opt_G.zero_grad()


        loss_idt_A = criterion_idt(G_AB(real_B), real_B) * cfg.lambda_B * cfg.lambda_identity
        loss_idt_B = criterion_idt(G_BA(real_A), real_A) * cfg.lambda_A * cfg.lambda_identity

        loss_G_AB = gan_loss(D_B(fake_B), True)
        loss_G_BA = gan_loss(D_A(fake_A), True)

        loss_cyc_A = criterion_cycle(rec_A, real_A) * cfg.lambda_A
        loss_cyc_B = criterion_cycle(rec_B, real_B) * cfg.lambda_B

        loss_G = loss_idt_A+loss_idt_B+loss_G_AB+loss_G_BA+loss_cyc_A+loss_cyc_B

        loss_G.backward()
        opt_G.step()

        # discriminator update
        set_requires_grad([D_A, D_B], True)
        opt_D.zero_grad()

        fake_A_buf = fake_A_pool.query(fake_A)
        fake_B_buf = fake_B_pool.query(fake_B)

        loss_D_A = (gan_loss(D_A(real_A), True) + gan_loss(D_A(fake_A_buf), False)) * 0.5
        loss_D_B = (gan_loss(D_B(real_B), True) + gan_loss(D_B(fake_B_buf), False)) * 0.5

        loss_D_A.backward()
        loss_D_B.backward()
        opt_D.step()

        # logging
        if i % cfg.print_freq == 0:
            print(f"epoch {epoch} step {i}  G {loss_G.item():.3f}  "
                  f"D_A {loss_D_A.item():.3f}  D_B {loss_D_B.item():.3f}  "
                  f"cyc {(loss_cyc_A + loss_cyc_B).item():.3f}")

    # end of epoch
    with torch.no_grad():
      fake = G_AB(fixed_A)
      rec = G_BA(fake)
      grid = torch.cat([fixed_A, fake, rec], dim=0)   # row 1: originals, row 2: Celtics, row 3: round trip
      save_image((grid + 1) / 2, sample_dir / f"epoch{epoch}.png", nrow=4)
    sched_G.step()
    sched_D.step()
    save_checkpoint(epoch)


'\ntest_ds = UnpairedDataset(cfg, phase="test")\nfixed_A = torch.stack([test_ds[i]["A"] for i in range(4)]).to(device)   # 4 fixed test images\n\n#switch gradient tracking on or off for a list of networks\ndef set_requires_grad(nets, flag):\n    for net in nets:\n        for p in net.parameters():\n            p.requires_grad = flag\n\ntrain_ds = UnpairedDataset(cfg, phase="train")\ntrain_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)\n\nsample_dir = ROOT / "samples"\nsample_dir.mkdir(exist_ok=True)\n\nstart_epoch = load_checkpoint() if cfg.continue_train else 1\ntotal_epochs = cfg.n_epochs + cfg.n_epochs_decay\n\nfor epoch in range(start_epoch, total_epochs + 1):\n    for i, batch in enumerate(train_loader):\n        real_A = batch["A"].to(device)\n        real_B = batch["B"].to(device)\n\n        # forward passes\n        fake_B = G_AB(real_A)     # other -> Celtics\n        rec_A  = G_BA(fake_B)     # back to other\n        fake_A

Phase 4: Evaluation

In [15]:
# evaluation

# load generators
ckpt = torch.load(Path(cfg.checkpoints_dir) / "latest.pth", map_location=device)
G_AB.load_state_dict(ckpt['G_AB'])
G_BA.load_state_dict(ckpt['G_BA'])
G_AB.eval()
G_BA.eval()

print("Loaded epoch", ckpt["epoch"])

results_dir = Path(cfg.results_dir)
results_dir.mkdir(parents=True, exist_ok=True)

test_ds = UnpairedDataset(cfg, phase="test")
n = min(cfg.num_test, len(test_ds.A_paths))

pairs = []
with torch.no_grad():
    for i in range(n):
        real_A = test_ds[i]["A"].unsqueeze(0).to(device)
        fake_B = G_AB(real_A)
        pair = torch.cat([real_A, fake_B], dim=0)
        save_image((pair + 1) / 2, results_dir / f"{test_ds.A_paths[i].stem}.png", nrow=2)
        pairs.append(pair)

overview = torch.cat(pairs[:16], dim=0)
save_image((overview + 1) / 2, results_dir / "overview.png", nrow=8)
print(f"Saved {n} results to {results_dir}")

Loaded epoch 50
Saved 50 results to /content/drive/MyDrive/cycleGAN_celtics_project/content/results
